# Isaac Sim Jupyter Notebook Tutorial

This notebook will demonstrate step by step how to load the IAI apartment a robot into the simulation environment.
 
> Select the kernel `Python 3.11.13 /mnt/dev-tools/isaac-sim-5.1/python.sh` if you are running in VScode.
>
> "Control + Enter" to execute the selected code cell. 

<!-- <button data-commandlinker-command="notebook:restart" class="jupyter-button">Force Stop</button> -->

## Start the GPU monitor and virtual desktop

Make sure it has at least 3000M free memory.

> Note: This tutorial does not start the ROS extension, so you can ignore the warning about ROS not being installed.

In [ ]:
from gpu_monitor import GPUMonitor
# Monitor GPU usage
gpu_monitor = GPUMonitor()

from utils import *
# Extract precompiled cache
run_script(CACHE_EXTRACT_CMD)
# Open Desktop in sidecar
display_desktop()

## Start SimulationApp

The application window is frozen and non-interactive, which is normal.

<div style="color:red">This will take some time, so give it a minute and wait for it to say <b>"SimulationApp Ready!"</b> before you go to next step.</div>

In [ ]:
from isaacsim import SimulationApp
from IPython.display import clear_output
import sys
import builtins

original_stdout = sys.stdout
original_stderr = sys.stderr

simulation_app = SimulationApp({
    "headless": False,
    # "hide_ui": True,
    "width": 1280,
    "height": 960,
    "renderer": "RaytracedLighting",
    "display_options": 3286,  # Setsimulation_app.update() display options to show default grid
})

# Fix the issue where notebook output is being hijacked by Isaac Sim.
sys.stdout = original_stdout
sys.stderr = original_stderr
clear_output(wait=True)
print('SimulationApp Ready!')

## Define the physical properties of the simulation environment

In [ ]:
from isaacsim.core.api import World

my_world = World(stage_units_in_meters=1.0,
                 physics_dt=1 / 200,
                 rendering_dt=8 / 200)
my_world.reset()

## Refresh View

Until now, we don't see any change in the app window, that's because it needs to be manually refreshed.

In [ ]:
from tqdm import tqdm

def refresh_view(steps=10):
    bar_format = "{l_bar}{bar}| {n_fmt}/{total_fmt} steps]"
    for i in tqdm(range(steps), desc="World Stepping", ncols=60, bar_format=bar_format):
        my_world.step(render=True)

refresh_view()

## Spawn environment USD to the world

The apartment USD is converted from the [iai apartment URDF](https://github.com/code-iai/iai_maps/blob/ros-jazzy/iai_apartment/urdf/apartment.urdf).

[Tutorial: import URDF](https://docs.isaacsim.omniverse.nvidia.com/5.1.0/importer_exporter/import_urdf.html#isaac-sim-app-tutorial-advanced-import-urdf)

In [ ]:
import os
from isaacsim.core.utils.prims import define_prim

prim = define_prim("/World/Apartment", "Xform")
asset_path = f"{os.getcwd()}/../usd/apartment/apartment.usd"
prim.GetReferences().AddReference(asset_path)

refresh_view()

## Change camera position

In [ ]:
import numpy as np
from isaacsim.core.utils import viewports

viewports.set_camera_view(eye=np.array([5, 0, 3]), target=np.array([0, 0, 0]))
refresh_view()

## Spawn Robot Anymal

The USD file for the Anymal robot is provided by Isaac Sim itself. 

You will see the robot collapse on the ground because no control commands have been sent to it yet.

In [ ]:
from isaacsim.robot.policy.examples.robots import AnymalFlatTerrainPolicy

robot = AnymalFlatTerrainPolicy(
    prim_path="/World/Anymal",
    name="Anymal",
    position=np.array([0, 0, 0.6]),
)
refresh_view(100)

## Add physics callback function to control the robot

The control command is a 3-element array, where the first value represents forward velocity, the second represents lateral (left/right) movement, and the third represents rotation. Value range is from -1 to 1.

In [ ]:
first_step = True
commands = [0.0, 0.0, 0.0]

def on_physics_step(step_size) -> None:
    global first_step, commands
    if first_step:
        robot.initialize()
        first_step = False
    else:
        robot.forward(step_size, commands)

my_world.add_physics_callback("physics_step", callback_fn=on_physics_step)

refresh_view(100)

## Restart Simulation and initialize robot

In [ ]:
first_step = True
my_world.reset()
commands = [0.0, 0.0, 0.0]
refresh_view(100)

## Send Control Commands

In [ ]:
# Forward
commands = [0.3, 0.0, 0.0]
refresh_view(steps=100)
# Move left
commands = [0.0, 0.3, 0.0]
refresh_view(steps=100)
# Turn around
commands = [0.0, 0.0, 0.8]
refresh_view(steps=100)
# Stop
commands = [0.0, 0.0, 0.0]
refresh_view(steps=100)

## Spawn Kitchen Objects

Load USD files from the repo

In [ ]:
from isaacsim.core.utils.prims import create_prim
from pxr import Gf
import random
import string

object_list = [
    "stretch",
    "Table049",
    "Toaster003",
]

for i in range(len(object_list)):
    obj = object_list[i]
    obj_prim = f"/World/{obj}"
    create_prim(
        usd_path=f"{os.getcwd()}/../usd/{obj}/{obj}.usd",
        prim_path=obj_prim,
        translation=Gf.Vec3d(2, -1 + i, 0.2)
    )

refresh_view(steps=100)

## Delete Kitchen Objects

In [ ]:
from isaacsim.core.utils.prims import delete_prim

for obj in object_list:
    delete_prim( f"/World/{obj}")

refresh_view()

## Running the simulation continuously

Once the following code cell is executed, it will enter an infinite loop. You can only terminate the entire program by restarting the kernel, the "Shutdown" button below is a shortcut, after which you'll need to rerun the previous code.

<button data-commandlinker-command="notebook:restart-clear-output" class="jupyter-button">Shutdown</button>

In [ ]:
# Reset Status
first_step = True
my_world.reset()

# A series of commands.
plans = [
    [0.3, 0.0, 0.0],
    [0.0, 0.0, 0.8],
    [0.3, 0.0, 0.0],
    [-0.3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
]

commands =  plans.pop(0)

# Each command is executed for N simulation frames.
N = 100
frame_count = 0

# while len(plans) != 0:
while simulation_app.is_running():
    my_world.step(render=True)
    if my_world.is_playing() and len(plans) != 0:
        frame_count += 1
        if frame_count % N == 0:
            commands = plans.pop(0)